# Lab 7 — Make something actually happen
### EP0418 · Topic 7 · 2 practical hours

**You need:** `ep0418.py` and the classroom sensor file; your group's project data once the track is chosen today.

---
Everything so far produced numbers. Today a number causes an **action**.

By the end of this session something must leave your notebook — a message, an
email, or a dashboard row that a named person could receive.

## Putting the assistant's code into a TODO cell

A TODO cell has a line ending in `...`. Replace **only that line** with the
assistant's code, and keep the name on the left (`s`, `clean`, `H`). Delete any
`import` or `read_csv` lines it adds; the data is already loaded. If it used a
different name for the result, add one line at the end to rename it, e.g.
`s = hourly_df`.

---
## How every lab in this module works

You will **not** write this code from scratch. Ask your AI assistant to write it.
Your job is the part it cannot do: deciding what the right answer looks like,
reading what it gave you, and proving it did what you asked.

Every prompt in these notebooks opens with the rules that matter for that task:
do not drop rows, show the row count before and after, treat -999 as missing,
explain the code in plain words. Keep those lines when you adapt a prompt, and
start your own prompts the same way.

At the foot of each task is a **verification contract** — a few `check.*` lines.
**You may not edit them.** The AI can write any implementation it likes; it cannot
satisfy these by accident.

If a check fails, read the message. It tells you what was found and what to try
next. Failing checks are the normal state of a lab in progress, not a sign you
are behind.

> **First: File → Save a copy in Drive, and work in your copy.** This page opened from GitHub, and changes here are not saved. Reopen your copy from Google Drive next time.

In [ ]:
import os, urllib.request
REPO = "https://raw.githubusercontent.com/StanleySP-eng/EP0418-labs/main/labs/"


def fetch(*names):
    """Download any course file that is not beside the notebook yet."""
    for name in names:
        if os.path.exists(name):
            continue
        try:
            urllib.request.urlretrieve(REPO + name, name)
            print("fetched", name)
        except Exception:
            print(f"Could not download {name}. Upload it with the folder icon on the left.")
            from google.colab import files
            files.upload()


fetch("ep0418.py")
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from ep0418 import check, show


def filled(*values):
    """True once every TODO slot above has been replaced. Otherwise says so."""
    if any(v is ... or v is None for v in values):
        print("A TODO above is still '...'. Fill it in, then run this cell again.")
        return False
    return True


def submit_prediction(team, lab, prediction, method):
    """Send one team's number to the Prediction League board."""
    import urllib.parse
    form = "https://docs.google.com/forms/d/e/1FAIpQLSdwpzHafC_rmPDdbK_cm6J8CihD-NSVS7ZBUyYCWIrS_MMJ0g/formResponse"
    fields = {'team': 'entry.911171418', 'lab': 'entry.1890998159', 'prediction': 'entry.959891957', 'method': 'entry.1997288724'}
    if not team or not method or prediction is None:
        print("Fill in TEAM, MY_PREDICTION and METHOD above, then run this cell again."); return
    entry = {"team": team, "lab": str(lab), "prediction": f"{float(prediction):.1f}", "method": method}
    if not form:
        print("The Prediction League is not switched on yet; tell your tutor your number."); return
    data = urllib.parse.urlencode({fields[k]: v for k, v in entry.items()}).encode()
    try:
        urllib.request.urlopen(urllib.request.Request(form, data=data, headers={"User-Agent": "EP0418"}), timeout=60)
        print(f"Entered: {team} predicts {float(prediction):.1f}. Check the scoreboard in a minute.")
    except Exception as e:
        print("The league did not answer:", e, "- try again in a minute.")

CSV = "smart_building_sensors.csv"     # or your group's project data
fetch(CSV)

df = pd.read_csv(CSV, parse_dates=["timestamp"])
df.loc[df.humidity_pct == -999, "humidity_pct"] = np.nan
df = df.drop_duplicates(subset="timestamp", keep="first").sort_values("timestamp")

Groups form this week, and each group chooses its project track. Task 0 is the
pre-lab: run the six track baselines at home, read the two questions, and bring
two favourites to the stand-up. The lab itself starts at Task 1.

## Task 0 — Six tracks, two favourites (pre-lab, at home)

The cells below run the baseline of every track in one place; each fetches its
own data. Run them in order. Each section ends with a line beginning
`RECORD THIS:`; that is the number for the table at the end of this task. Do not
edit anything. You are looking, not building.

In [ ]:
import os, urllib.request
import pandas as pd, numpy as np

if not os.path.exists("ep0418.py"):                      # the course library, fetched once
    urllib.request.urlretrieve("https://raw.githubusercontent.com/StanleySP-eng/EP0418-labs/main/labs/ep0418.py", "ep0418.py")
from ep0418 import show

PACK = {"smart_money_ohlcv.csv": "track1_smart_money", "smart_building_sensors.csv": "track2_smart_building",
        "sg_rainfall_daily.csv": "track3_smart_farm",
        "sg_taxi_hourly.csv": "track4_smart_city", "smart_self_daily.csv": "track5_smart_self",
        "example_data.csv": "track6_free_form"}
RAW = "https://raw.githubusercontent.com/StanleySP-eng/EP0418-labs/main/starter_packs/"

def find(name):
    # beside the notebook (or unzipped), inside a track folder, or fetched from the course repository
    for cand in [name] + [os.path.join(d, name) for d in os.listdir(".") if os.path.isdir(d)]:
        if os.path.exists(cand):
            return cand
    try:
        import urllib.request
        urllib.request.urlretrieve(RAW + PACK[name] + "/" + name, name)
    except Exception:
        from google.colab import files; files.upload()
    return name

RECORD = {}
def record(track, label, value):
    RECORD[track] = (label, value)
    print(f"\nRECORD THIS:  {label} = {value}")

## Track 1 — Smart Money (STI ETF daily bars)

**The data.** One row per trading day for an exchange-traded fund that follows
the Singapore share index: the price it opened and closed at, the high and low,
and how many units changed hands. About 250 rows a year, because markets close at
weekends and on holidays. The prices are synthetic, modelled on the real fund, so
the file can be shared.

**The baseline.** *Buy on day one, sell on the last day, do nothing in between.*
Any trading rule a project builds has to beat that, after costs. Most do not.

**What the code does.** Loads the file, sorts it by date, divides the last closing
price by the first to get the total return, and records it. Then it draws the
closing price.

**Look for:** how much of the rise comes in long steady climbs, and how much in a
few sharp moves. A rule that is out of the market during the sharp moves loses to
buy-and-hold even if it is right most days.

In [ ]:
mny = pd.read_csv(find("smart_money_ohlcv.csv"), parse_dates=["Date"]).sort_values("Date")
print(f"rows {len(mny):,}   period {mny.Date.min().date()} -> {mny.Date.max().date()}   cadence: trading days")
bh_total = mny.Close.iloc[-1] / mny.Close.iloc[0] - 1
print(f"buy-and-hold total return: {bh_total:+.1%}   (buy on day 1, sell on the last day, no trading)")
record("1 Smart Money", "buy-and-hold return", f"{bh_total:+.1%}")
show.series(mny, col="Close", title="Track 1: closing price, one point per trading day")

**What you should see:** a price that roughly doubles over three and a half years,
with two or three dips of ten to fifteen per cent on the way. The number to
record is the buy-and-hold return, about +99%. Ask yourself: would a rule that
sells on every dip and buys back later have done better, or just missed the
recovery?

## Track 2 — Smart Building (classroom sensors, every 5 minutes)

**The data.** Sixty days of readings from a classroom: temperature, humidity,
CO2, energy use and how many people are in the room, every five minutes. It is
the file you profiled in Lab 1, −999s and all. It is synthetic with six faults
planted in it on purpose, so every student sees the same problems.

**The baseline.** *Alert whenever CO2 goes above 1000 ppm.* The simplest possible
alarm, and the one most buildings actually use.

**What the code does.** Loads the file, marks every reading above 1000 as an
alert, counts them, and counts how many fired while the occupancy column said the
room was empty. Then it draws CO2 with the threshold as a dashed line.

**Look for:** whether the readings sit at the same level all sixty days, or creep
upward towards the end.

In [ ]:
b = pd.read_csv(find("smart_building_sensors.csv"), parse_dates=["timestamp"])
print(f"rows {len(b):,}   period {b.timestamp.min()} -> {b.timestamp.max()}   cadence: 5 minutes")
alert = b.co2_ppm > 1000
print(f"fixed-threshold alerts (CO2 > 1000 ppm): {alert.sum()} of {len(b):,} readings; "
      f"{(alert & (b.occupancy == 0)).sum()} fired while the room was empty")
record("2 Smart Building", "alerts from the fixed threshold", int(alert.sum()))
show.series(b, col="co2_ppm", title="Track 2: CO2 every 5 minutes, with the alert threshold", hline=1000)

**What you should see:** a daily rhythm of peaks on school days and quiet
weekends, and in the last three weeks the whole line sitting higher, so that it
crosses the dashed threshold more and more often. About 1,890 alerts, twenty of
them in an empty room. The room did not change; the sensor drifted. A fixed
threshold cannot tell the difference, and that is the problem this track is
about.

## Track 3 — Smart Farm (NEA daily rainfall)

**The data.** Real daily rainfall from the National Environment Agency, averaged
across the island, for sixty days. One row per day. Nothing is planted in it; it
has whatever faults real data has.

**The baseline.** A farm has to decide each morning whether to irrigate. The
simplest rule is *do today what yesterday needed*: if yesterday was dry, water
today. The code calls a day "needs water" when the island-wide mean is under one
millimetre.

**What the code does.** Loads the file, marks each dry day, shifts that column by
one day to make yesterday's answer, and counts how often yesterday's answer was
right for today. It also counts the two ways of being wrong: watering a wet day
(wasted) and not watering a dry one (missed). Then it draws the rainfall with the
one-millimetre line.

**Look for:** whether dry days come in runs or alternate with wet ones. A rule
that copies yesterday only works when the weather has runs.

In [ ]:
r = pd.read_csv(find("sg_rainfall_daily.csv"), parse_dates=["date"]).sort_values("date")
print(f"rows {len(r)}   period {r.date.min().date()} -> {r.date.max().date()}   cadence: daily")
needs = r.rain_mm_mean < 1.0                       # the field needs water
yesterday = needs.shift(1)
m = yesterday.notna()
correct = (yesterday[m].astype(bool) == needs[m]).mean()
print(f"policy 'water if yesterday needed water': right on {correct:.1%} of days "
      f"({int((needs[m] & ~yesterday[m].astype(bool)).sum())} missed waterings, "
      f"{int((~needs[m] & yesterday[m].astype(bool)).sum())} wasted)")
record("3 Smart Farm", "days 'yesterday's value' was right", f"{correct:.1%}")
show.series(r, col="rain_mm_mean", title="Track 3: island-wide mean rainfall per day", hline=1.0)

**What you should see:** rainfall that jumps about from day to day, with many
days near zero and a few very wet ones. The rule is right less than half the
time, about 44%, which is worse than a coin. That is the point of this track: in
Singapore, yesterday says little about today, so a project here has to find
something that does, and a forecast that is wrong costs either water or a crop.

## Track 4 — Smart City (taxi availability, about hourly)

**The data.** Real readings from the Land Transport Authority's feed: how many
taxis were available island-wide, roughly once an hour for eight weeks. The
timestamps are not on the hour, and some hours are missing. This is the dataset
Labs 3 and 6 use.

**The baseline.** *Guess the same hour last week.* Called seasonal-naive, because
it assumes the weekly pattern repeats.

**What the code does.** Puts the readings onto a regular hourly grid (you learn
how in Lab 3), shifts the series by 168 hours to make last week's value, and
measures the average size of the miss, the mean absolute error, over the hours
where both exist. Then it draws the hourly series.

**Look for:** the daily cycle, the weekly cycle on top of it, and whether any day
breaks the pattern.

In [ ]:
t = pd.read_csv(find("sg_taxi_hourly.csv"), parse_dates=["timestamp"])
print(f"rows {len(t):,}   period {t.timestamp.min()} -> {t.timestamp.max()}   cadence: about hourly (irregular)")
s = t.set_index("timestamp").taxi_count.resample("h").mean()
sn = s.shift(168); m = s.notna() & sn.notna()
mae = (s - sn)[m].abs().mean()
print(f"seasonal-naive MAE (same hour last week): {mae:,.1f} taxis   (mean count {s.mean():,.0f})")
record("4 Smart City", "seasonal-naive MAE, taxis", round(float(mae), 1))
show.series(s, title="Track 4: taxis available, hourly")

**What you should see:** a strongly repeating daily wave, lower at night and
higher in the evening, with weekends visibly lower. The baseline misses by about
500 taxis an hour on an average of about 2,400: roughly a fifth. Beating that
means explaining what is *not* weekly: weather, events, holidays.

## Track 5 — Smart Self (wearable, one row per day)

**The data.** Six months of one person's daily steps, resting heart rate and
sleep, as a smartwatch exports them. Synthetic, with an illness, some days the
watch was not worn, and a few impossible readings built in. If you later use your
own export, you anonymise it first.

**The baseline.** *Tomorrow will be like the average of the last seven days.* A
rolling mean.

**What the code does.** Loads the file, puts it on a daily grid so that absent
dates show as gaps, computes the seven-day rolling mean shifted forward one day,
and measures the mean absolute error against the real steps. Then it draws the
steps.

**Look for:** days at exactly zero, and gaps where there is no row at all. They
are two different kinds of "no data", and only one of them is a day the person
did not move.

In [ ]:
w = pd.read_csv(find("smart_self_daily.csv"), parse_dates=["date"]).sort_values("date")
print(f"rows {len(w)}   period {w.date.min().date()} -> {w.date.max().date()}   cadence: daily "
      f"({(w.date.max() - w.date.min()).days + 1 - len(w)} dates absent)")
d = w.set_index("date").asfreq("D")
pred = d.steps.rolling(7).mean().shift(1); m = pred.notna() & d.steps.notna()
mae = (d.steps[m] - pred[m]).abs().mean()
print(f"rolling 7-day mean baseline: MAE {mae:,.0f} steps   (mean daily steps {d.steps.mean():,.0f})")
record("5 Smart Self", "rolling 7-day MAE, steps", int(round(mae)))
show.series(d.steps, title="Track 5: daily steps (gaps are days with no row)")

**What you should see:** steps bouncing around 8,000 a day, a fortnight in March
where they fall to a third, eleven days sitting at exactly zero, two spikes above
38,000, and three small breaks in the line. The baseline misses by about 2,800
steps on an average of 7,850. The zero days drag the rolling mean down for the
week after each one, which is why they matter.

## Track 6 — Free-form (the example file; yours will differ)

**The data.** Whatever you bring. This cell runs on a demo file, one value per
day for a year, so that the feasibility check works out of the box. Three worked
examples live in the Track 6 folder: Steam player counts, football results and
Formula 1 lap times.

**The baseline.** You define it in your proposal. The code shows the two most
common starting points: *same as yesterday* (naive) and *same as a week ago*
(seasonal-naive), and scores both.

**What the code does.** Loads the file, builds each guess by shifting the series,
and measures the mean absolute error of each. Then it draws the series.

**Look for:** whether a weekly guess beats a daily one. If it does, the data has
a weekly rhythm, and that is the first thing a project on it should use.

In [ ]:
e = pd.read_csv(find("example_data.csv"), parse_dates=["timestamp"]).sort_values("timestamp")
print(f"rows {len(e)}   period {e.timestamp.min().date()} -> {e.timestamp.max().date()}")
x = e.set_index("timestamp")["value"]
for name, p in [("naive (last value)", x.shift(1)), ("seasonal-naive (7 back)", x.shift(7))]:
    m = p.notna() & x.notna()
    print(f"{name:26s} MAE {(x[m] - p[m]).abs().mean():.3f}")
m = x.shift(1).notna()
record("6 Free-form", "naive MAE on the example file", round(float((x[m] - x.shift(1)[m]).abs().mean()), 3))
show.series(x, title="Track 6: the example file")

**What you should see:** a series with a clear weekly wobble, and the seasonal
guess beating the daily one by a wide margin (about 9 against 14). A free-form
project needs a dataset where a baseline this simple can be stated and scored; if
you cannot say what "same as last week" would mean for your data, it is not ready.

## The six numbers for your sheet

Copy this table into the Lab 2 sheet. Then answer the question that actually
shapes your shortlist: which of these would be hardest to beat, and why?

In [ ]:
print(f"{'Track':18s} {'Baseline':40s} {'Number'}")
for track, (label, value) in RECORD.items():
    print(f"{track:18s} {label:40s} {value}")

Fill this in from the `RECORD THIS` lines.


| Track | Rows | What period | Baseline number | Would I enjoy 14 weeks of this? |
|---|---|---|---|---|
| 1 Smart Money | | | | |
| 2 Smart Building | | | | |
| 3 Smart Farm | | | | |
| 4 Smart City | | | | |
| 5 Smart Self | | | | |
| 6 Free-form | | | | |

**The six starter notebooks, one per track.** Open the two you like most.

| Track | Starter notebook (opens in Colab; then File > Save a copy in Drive) |
|---|---|
| 1 Smart Money | [Track1_Smart_Money_Starter.ipynb](https://colab.research.google.com/github/StanleySP-eng/EP0418-labs/blob/main/starter_packs/track1_smart_money/Track1_Smart_Money_Starter.ipynb) |
| 2 Smart Building | [Track2_Smart_Building_Starter.ipynb](https://colab.research.google.com/github/StanleySP-eng/EP0418-labs/blob/main/starter_packs/track2_smart_building/Track2_Smart_Building_Starter.ipynb) |
| 3 Smart Farm | [Track3_Smart_Farm_Starter.ipynb](https://colab.research.google.com/github/StanleySP-eng/EP0418-labs/blob/main/starter_packs/track3_smart_farm/Track3_Smart_Farm_Starter.ipynb) |
| 4 Smart City | [Track4_Smart_City_Starter.ipynb](https://colab.research.google.com/github/StanleySP-eng/EP0418-labs/blob/main/starter_packs/track4_smart_city/Track4_Smart_City_Starter.ipynb) |
| 5 Smart Self | [Track5_Smart_Self_Starter.ipynb](https://colab.research.google.com/github/StanleySP-eng/EP0418-labs/blob/main/starter_packs/track5_smart_self/Track5_Smart_Self_Starter.ipynb) |
| 6 Free-form | [Track6_Free_Form_Starter.ipynb](https://colab.research.google.com/github/StanleySP-eng/EP0418-labs/blob/main/starter_packs/track6_free_form/Track6_Free_Form_Starter.ipynb) |

### The two questions that actually decide it

**1. How hard is the baseline?**
Some are much harder than they look. Buy-and-hold returned +99.4% over the sample
period — beating that after costs is genuinely difficult. A fixed CO₂ threshold,
by contrast, is easy to beat once you handle drift.

Neither is the "right" choice. But choose knowing which you picked.

**2. Where will your automated response go?**
Every project must trigger something real. Who receives it? If you
cannot name a plausible recipient for a track, that is a warning sign.

### Two favourites

In the cell below, write the two tracks you liked most and why. No commitment:
this is the shortlist you bring to your group today.

**Favourite 1:** *(track · why · its baseline number · who would receive the response)*

**Favourite 2:** *(track · why · its baseline number · who would receive the response)*

**The one I would find hardest, and why:**

## Task 1 — Write the sentence first

Fill in all three blanks before you write any code.

> When **____**, the system will **____**, and **____** will receive it.

If you cannot name a specific recipient, fix that before continuing — you do not
yet have a decision system.

**When:**

**The system will:**

**Received by:**

**Cost of a false alarm, and who pays it:**

## Task 2 — Build the rule, with debouncing

A naive rule fires on every single reading that crosses the line. Real systems
require the condition to **hold**.

> Flag periods where `co2_ppm` has been above 1000 for at least 15 consecutive
> minutes (3 consecutive 5-minute readings), rather than flagging single readings.
> Do not drop rows. Say how many periods you flag.

In [ ]:
# TODO(you): `fires` = a boolean Series, True only when the condition has held
fires = ...

naive_count = (df.co2_ppm > 1000).sum()
if filled(fires):
    print(f"without debouncing: {naive_count:,} readings above the line")
    print(f"with debouncing   : {fires.sum():,} readings during a held condition")
    starts = fires & ~fires.shift(1, fill_value=False)
    print(f"                    {starts.sum():,} separate alert events")
    show.alerts(df, 1000, col="co2_ppm", naive=df.co2_ppm > 1000, debounced=fires,
                title="Where the alerts fire: every crossing, and only held conditions")

**Expected output** *(computed from the data file when this notebook was built)*: 1,873 readings above the line without debouncing; with a three-reading hold, 1,538 readings in 117 separate events. If your debounced count is higher than the naive one, the rule is averaging rather than requiring.

**What you should see:** the top row of ticks is dense wherever the line dips over
the threshold, including single readings that cross and come straight back. The
second row is sparse: it starts only once the condition has held. The gap between
the two rows is the number of messages your recipient did not have to read.

## Task 3 — Rate-limit it

Even debounced, a bad afternoon can send ninety messages. Cap it at **one alert
per hour per condition**.

An *alert event* is one row per alert, with the columns `timestamp`, `co2_ppm`
and (later) `message`.

> `fires` is a boolean Series aligned to `df`. Build `alerts`: a DataFrame with
> one row per alert, columns `timestamp` and `co2_ppm`, keeping at most one
> alert per clock hour. Show how many alerts before and after that limit.

In [ ]:
# TODO(you): `alerts` = a DataFrame of alert events, at most one per hour
alerts = ...

if filled(alerts):
    print(f"after rate limiting: {len(alerts)} alerts over {(df.timestamp.max()-df.timestamp.min()).days} days")

**Expected output** *(computed from the data file when this notebook was built)*: about 106 alerts over sixty days with at most one per hour: roughly two a day, which a person can act on.

## Task 4 — Write the message a human will read

Compare:

- `"ALERT"`
- `"CO₂ 1240 ppm in Lab 4, sustained 20 min — check ventilation"`

The second names the value, the place, the duration, and the action. Write yours
to that standard.

In [ ]:
# TODO(you): build `messages` — a list of human-readable alert strings
messages = ...

if filled(messages):
    for m in messages[:5]:
        print(m)

## Task 5 — Actually send one

Pick **one** of the three and make it work. All are free.

**Telegram** (most students choose this):

1. In Telegram, message `@BotFather`, send `/newbot`, follow the prompts, and
   copy the token it gives you (a long string with a colon in it).
2. Send your new bot any message from your own account.
3. In a browser, open `https://api.telegram.org/bot<TOKEN>/getUpdates` with your
   token in place of `<TOKEN>`. Find `"chat":{"id": ...}`; that number is your
   `chat_id`.
4. Send a message from Python:
   `requests.get(f"https://api.telegram.org/bot{TOKEN}/sendMessage", params={"chat_id": CHAT_ID, "text": message})`

**Never leave your token in a notebook you submit.** Put it in Colab's Secrets
(the key icon on the left) or in a cell you delete before submitting.

**Email**: Python's built-in `smtplib`. Good when the recipient is not you.

**Dashboard**: Streamlit, deployed free to a public URL. Good when the response is
"a human should come and look".

In [ ]:
# TODO(you): send at least one real notification.
#            Keep evidence — the API response, or the list of messages sent.
sent = ...

# ---- verification contract — do not edit ----
check.response_fired(sent, minimum=1)
check.report()

---
## Predict: What will the next Cat A COE premium be? (10 minutes, teams of two or three)

*The Prediction League, week 6. Same team name as last time.*

What will the Category A premium be at the next COE bidding exercise? You have every round since 2010. Lab 7 was about deciding with a rule you can defend. The baseline, doing nothing clever, is on the board already:
the premium at the most recent round in the file. The target week's dates are on the scoreboard, and last week's
result is there too: look at who was closest and what they said they did.

In [ ]:
fetch("coe_bidding.csv")
coe = pd.read_csv("coe_bidding.csv")
cat_a = coe[coe.vehicle_class == "Category A"]
print(f"the last five Cat A premiums : {cat_a.premium.tail(5).tolist()}")
print(f"the most recent              : S${cat_a.premium.iloc[-1]:,.0f}   <- the baseline")
print(f"change over the last three   : S${cat_a.premium.iloc[-1] - cat_a.premium.iloc[-4]:+,.0f}")

**Step by step.** Open Gemini (spark icon, bottom right), **+** for a new chat,
paste, Enter:

> I am new to this. Answer in three plain sentences, no headings, no code.
> What will the Category A premium be at the next COE bidding exercise? What would you base the guess on, using what this week's lab taught,
> and why might it still be wrong?

Read the three sentences. Decide with your team: the baseline as printed, one of
the other numbers printed above, or a number Gemini computes for you (rules first,
**+ Code**, paste, run, read). Fill in the three lines below, for example
`TEAM = "Team Rocket"`, `MY_PREDICTION = 128000`, `METHOD = "the baseline, nudged
for ..."`, and run the cell. Ignore Gemini's blue suggestion chips.

In [ ]:
TEAM = ""                 # TODO(you): the same team name as last week
MY_PREDICTION = None      # TODO(you): your number, e.g. 128000
METHOD = ""               # TODO(you): one line on how you got it

submit_prediction(TEAM, 7, MY_PREDICTION, METHOD)

---
## Read before you run

An assistant was asked to fire only when CO2 has been above 1000 ppm for three consecutive readings. It wrote:

The code is in the cell below, commented out. It came from an AI assistant in
someone else's chat; you are not asking Gemini for it. **Do not run it yet.** Read
it, answer the two questions below, and only then remove the `#` signs and run it
to see whether you were right.

Is “the average of three readings is above 1000” the same as “all three readings are above 1000”? Give three numbers that make this rule fire when it should not.

Write down what the rule counts over the full period compared with the correct debounce.

<details><summary>Hint 1: where to look</summary>

Can the average of three readings be above 1000 while one of them is below it?

</details>
<details><summary>Hint 2: closer to the answer</summary>

996, 1010 and 1003 average 1003: the rule fires, but the condition never held. Over sixty days the mean rule fires 1,855 times against 1,538 for a genuine three-reading hold.

</details>

**After you have written your answer, and not before:** select the cell, click the
pencil-and-spark icon at its top right and choose **Explain code**. Did Gemini notice
the fault? It often does not. That is the point: an explanation that reads well is
not evidence the code is right.

**To fix it:** choose **Transform code** on the same cell and say what to change, in
plain words. Colab shows the change as a diff before it is applied. Read the diff. If
it touches a `check.` line, reject it.

In [ ]:
# An AI assistant wrote this for a classmate. It is already here: there is nothing to ask Gemini for.
# 1. Read it. Answer the two questions in the cell below, in your own words.
# 2. Then select this cell, click the pencil-and-spark icon, choose Explain code. Did Gemini see the fault?
# 3. Only then remove the # signs and run it, to see whether you were right.
# fires = df.co2_ppm.rolling(3).mean() > 1000

**What is wrong with it:**

**What it would produce, and what it should produce:**

---
## Quiz yourself before you leave (5 minutes)

The checks tell you whether the code is right. They do not tell you whether
**you** know what it did. Paste this into the Gemini chat:

> Quiz me on this lab. Ask five short questions, one at a time, in plain words, on these ideas: the three blanks of a decision sentence: when, what, who; what debouncing and rate limiting prevent; what makes an alert message actionable; what counts as evidence that a response arrived; why an alert nobody can act on is worse than none. Take the examples from the code and outputs in this notebook. Wait for my answer. Then say in one sentence whether I am right, and why. Do not give the next question until I answer. After the fifth, give my score out of 5 and the one idea I should revise, in one line.

Gemini asks in the chat panel; you answer in the chat box; it marks you and asks
the next. Ignore the suggestion chips under its messages. Answer in your own
words, without looking back until it has marked you.

Then write your score and the idea to revise below. Your tutor asks for them in
the plenary.

**Quiz score:** ___ / 5

**The idea I should revise, in my own words:**

---
## Before you leave

- A real notification has arrived somewhere you can show your tutor.
- **Mistake caught** (below): how many alerts did the naive rule fire, versus
  your debounced and rate-limited version? What would the naive version have cost
  the recipient?
- A **Decisions** line: your threshold, debounce and rate limit, who receives the
  alert, and why.

**Mistake caught:** *(what I expected · what was there · how I found it)*

**Decision:** *(what I decided · why · the evidence)*